# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL and includes multiple record sets representing survey responses, logistic regression outputs, and socio-demographic predictors for adoption of indigenous and modern knowledge in rangeland management (Northern Kenya).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Access top-level metadata attributes
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")
print(f"Spatial Coverage: {metadata.spatialCoverage}")

## 2. Data Overview
Review available record sets, fields, and their unique `@id`s.

The dataset contains structured survey data and logistic regression outputs. In Croissant, record sets represent top-level tables or data groups. All entities—record sets, fields, and columns—are referenced by their `@id`.

In [ ]:
# Display available record sets and their @id
record_sets = dataset.record_sets
print("Available record sets and their @id:")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# For each record set, review available fields
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', 'Unnamed')})")
    print("Fields and their @id:")
    for field in rs.get('fields', []):
        print(f"  - @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis.

We'll select two representative record sets by their `@id`: one for household survey results, and one for ordered logistic regression outputs. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record sets by their @id
# (Replace with actual @id values from previous cell)
main_survey_rs_id = None
regression_outputs_rs_id = None

for rs in record_sets:
    if 'survey' in rs.get('name', '').lower():
        main_survey_rs_id = rs['@id']
    if 'regression' in rs.get('name', '').lower():
        regression_outputs_rs_id = rs['@id']

# Proceed if the record sets are discovered
record_set_ids = [main_survey_rs_id, regression_outputs_rs_id]
dataframes = {}

for rs_id in record_set_ids:
    if rs_id:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"\nColumns in {rs_id}:")
        print(dataframes[rs_id].columns.tolist())
        print(dataframes[rs_id].head())
    else:
        print(f"Record set not found for one of the selected types.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filter records based on numeric field threshold
- Normalize values
- Group data by categorical field

All fields are referenced by their `@id`.

In [ ]:
# Example: Analyze log likelihood in regression outputs

# Identify numeric and group field IDs from previous overview
numeric_field_id = None
group_field_id = None
if regression_outputs_rs_id:
    df = dataframes[regression_outputs_rs_id]
    # Try matching log likelihood and location fields to @id
    for rs in record_sets:
        if rs['@id'] == regression_outputs_rs_id:
            for field in rs.get('fields', []):
                if 'log likelihood' in field.get('name', '').lower():
                    numeric_field_id = field['@id']
                if 'ward' in field.get('name', '').lower() or 'location' in field.get('name', '').lower():
                    group_field_id = field['@id']
    # If columns are named with @id, use their exact @id
    if numeric_field_id in df.columns:
        threshold = -50
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by location/ward if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print(f"Group field {group_field_id} not found in DataFrame.")
    else:
        print(f"Numeric field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We plot normalized log likelihood by ward/location if available.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt

# Ensure fields are set
if regression_outputs_rs_id and numeric_field_id and group_field_id:
    df = dataframes[regression_outputs_rs_id]
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in df.columns and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        df.groupby(group_field_id)[norm_col].mean().plot(kind='bar')
        plt.title(f"Mean Normalized Log Likelihood by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel("Normalized Log Likelihood")
        plt.show()
    else:
        print("Required columns for visualization not found.")
else:
    print("Visualization setup incomplete due to missing field IDs.")

## 6. Conclusion
In this notebook, we loaded, explored, and analyzed the FAIR² dataset using the `mlcroissant` library referencing all entities via their `@id`.

- Used Croissant schema URL to load metadata and records programmatically
- Reviewed available record sets, fields, and column `@id`s
- Extracted main survey and regression outputs into DataFrames
- Performed EDA including filtering, normalization, and grouping
- Visualized distribution of normalized log likelihood by ward/location

**Key Takeaways:**
- Survey highlights biases in gender, income, and age reporting
- Regression captures household adoption variability, but missing values limit generalization
- Visualization reveals regional variation in knowledge adoption predictors

Further analysis may focus on deeper subgroup analysis, missing value imputation, and policy relevance.